# Lab RAG RuangKarya: empat kasus yang sering bikin jawaban salah

**Praktik terarah · sekitar 65 menit**

RuangKarya adalah penyedia ruang kerja **fiktif**. Kamu bertugas menjawab pertanyaan pelanggan dan petugas dengan bantuan dokumen kebijakan serta manual perangkat. Dokumen yang benar bisa saja ada, tetapi sistem RAG tetap memberi jawaban yang salah. Di lab ini, kamu akan mencari **di langkah mana masalahnya muncul**, mencoba satu perbaikan, lalu mengecek hasilnya.

| Kasus | Gejala | Perbaikan yang diuji |
| --- | --- | --- |
| 1 | Aturan lama 48 jam ikut muncul | Saring dokumen berdasarkan status |
| 2 | Kode `R17` terpisah dari artinya | Jangan potong satu baris tabel |
| 3 | `R17` KD-410 tertukar dengan KD-420 | Bandingkan dense, BM25, dan RRF; cek model perangkat |
| 4 | Dokumen kartu hilang ada, tetapi angka biayanya tidak ada | Cek bukti; jangan menebak angka (abstain) |

**Cara membaca hasil:** **SIMULASI TERKENDALI** adalah contoh yang sengaja kami atur agar masalahnya terlihat. **HASIL RUN** berasal dari kode yang kamu jalankan: OpenAI membuat embedding, lalu Pinecone mencari passage dengan dense search atau BM25. RRF menggabungkan urutan hasil kedua metode tersebut. Anggap hasil run itulah pengukuranmu; jangan memakai urutan simulasi sebagai bukti bahwa suatu metode lebih baik.

**Catatan untuk pengajar:** hasil ranking dari API asli belum dipilotkan di materi ini. Jalankan sekali dengan API key sebelum kelas, lalu catat ranking dan waktunya.

## Mulai dari data — 10 menit

Ikuti langkah setup di `README.md`: salin `.env.example` menjadi `.env`, lalu isi OpenAI API key dan Pinecone API key di sana. Notebook memuat `.env` saat dijalankan. Kamu tidak perlu mengetik key di terminal atau menaruhnya di sel notebook.

Model `text-embedding-3-small` mengubah teks dokumen dan pertanyaan menjadi vektor. Pinecone menyimpan passage beserta vektornya, lalu menjalankan dense search dan BM25. Saat pertama kali dijalankan, teks korpus fiktif dikirim ke OpenAI API; run berikutnya memakai cache embedding. Pencarian Pinecone tetap memerlukan internet.

Dalam lab ini kamu berperan sebagai **petugas**, jadi dokumen berlabel `publik` dan `petugas` boleh dipakai. Kalau ini aplikasi untuk pelanggan, dokumen `petugas` harus disaring sebelum retrieval. Jawaban acuan dalam kode hanya untuk mengecek hasil lab, bukan untuk memilih jawaban secara otomatis.

In [ ]:
import os
import re
import time
import hashlib
import json
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone, DenseVectorQuery, TextQuery

from rag_helpers import (
    embed_cached, evidence_rank, load_corpus,
    make_passage, passage_per_section, show_results,
)

ROOT = Path.cwd().resolve()
if not (ROOT / "corpus").is_dir():
    raise FileNotFoundError("Buka Jupyter dari folder v3 yang berisi corpus/")
if not (ROOT / ".env").is_file():
    raise FileNotFoundError("Salin .env.example menjadi .env, lalu isi dua API key dan namespace")
load_dotenv(ROOT / ".env", override=True)
for key in ("OPENAI_API_KEY", "PINECONE_API_KEY", "PINECONE_INDEX", "PINECONE_NAMESPACE"):
    if not os.getenv(key):
        raise RuntimeError(f"Isi {key} di file .env, lalu jalankan ulang sel ini")
if os.environ["PINECONE_NAMESPACE"] == "ruangkarya-v3-nama-kamu":
    raise RuntimeError("Ganti PINECONE_NAMESPACE dengan nama unik milikmu di file .env")

CASES = {
    "C1": {"question": "Apakah biaya kembali jika saya membatalkan 12 jam sebelum mulai?", "source": "PEMBATALAN", "evidence": "kurang dari 24 jam sebelum waktu mulai tidak mendapat pengembalian"},
    "C2": {"question": "Apa arti R17 pada terminal KD-410?", "source": "KD410-ERROR", "evidence": "Kode: R17; Arti: Pembaca kartu akses tidak tersambung"},
    "C3": {"question": "Apa arti R17 pada terminal KD-420?", "source": "KD420-ERROR", "evidence": "Sensor posisi pintu belum tersinkron"},
    "C4": {"question": "Berapa biaya mengganti kartu akses yang hilang?", "source": None, "evidence": None},
}

documents = load_corpus(ROOT / "corpus")
ALLOWED_ACCESS = {"publik", "petugas"}  # Peran yang dipakai lab ini: petugas.
active_docs = [
    doc for doc in documents
    if doc["status"] == "berlaku" and doc["access"] in ALLOWED_ACCESS
]
archive_docs = [doc for doc in documents if doc["status"] != "berlaku"]
assert len(active_docs) == 10 and len(archive_docs) == 1
all_passages = passage_per_section(documents)
active_passages = [
    p for p in all_passages
    if p["status"] == "berlaku" and p["access"] in ALLOWED_ACCESS
]
assert all(p["access"] in ALLOWED_ACCESS for p in active_passages)

# Contoh pemotongan sengaja buruk: 'R17' dan artinya menjadi dua passage.
good_r17 = next(p for p in active_passages if p["source_id"] == "KD410-ERROR" and "Kode: R17" in p["text"])
kd410_doc = next(doc for doc in active_docs if doc["source_id"] == "KD410-ERROR")
cut = good_r17["text"].index("Arti: ") + len("Arti: ")
bad_r17 = [
    make_passage(kd410_doc, "potong-1", "tabel terpotong", good_r17["text"][:cut]),
    make_passage(kd410_doc, "potong-2", "tabel terpotong", good_r17["text"][cut:]),
]

print("Dokumen aktif:", len(active_docs), "| arsip:", len(archive_docs))
print("Passage aktif:", len(active_passages), "| pertanyaan kasus:", len(CASES))
print("Arsip:", [(doc["source_id"], doc["version"], doc["status"]) for doc in archive_docs])

### Buat embedding dan isi index Pinecone

Di sini, *passage* berarti potongan dokumen yang bisa dicari. Kita menyiapkan passage dari dokumen aktif, arsip untuk contoh kesalahan, dua potongan tabel yang sengaja dibuat buruk, dan lima pertanyaan. Semuanya dikirim dalam **satu permintaan embedding OpenAI**. Passage beserta vektornya lalu dimasukkan ke satu index Pinecone. Dense search memakai vektor; BM25 memakai teks. Jika model dan teksnya tidak berubah, embedding dibaca dari `.cache/` pada run berikutnya. Query Pinecone tetap memakai jaringan.

**Tebak dulu:** mengapa dokumen dan pertanyaan perlu di-embed dengan model yang sama? Mengapa kita tetap perlu menandai passage arsip walaupun semuanya masuk ke index?

In [ ]:
EMBED_MODEL = "text-embedding-3-small"
embedding_items = all_passages + bad_r17
embedding_texts = [p["text"] for p in embedding_items] + [case["question"] for case in CASES.values()]
vectors, embedding_info = embed_cached(embedding_texts, EMBED_MODEL, ROOT / ".cache", OpenAI)
passage_vectors = {p["passage_id"]: vectors[i] for i, p in enumerate(embedding_items)}
query_vectors = {case_id: vectors[len(embedding_items) + i] for i, case_id in enumerate(CASES)}
passage_by_id = {p["passage_id"]: p for p in embedding_items}
RUN_ID = hashlib.sha256(
    json.dumps({"model": EMBED_MODEL, "passages": [
        (p["passage_id"], p["text"], p["status"], p["access"], p["model"])
        for p in embedding_items
    ]},
               ensure_ascii=False).encode("utf-8")
).hexdigest()[:16]
INDEX_NAME = os.environ["PINECONE_INDEX"]
NAMESPACE = os.environ["PINECONE_NAMESPACE"]

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
if not pc.indexes.exists(INDEX_NAME):
    print("Membuat index Pinecone. Langkah pertama ini bisa memerlukan beberapa menit...")
    pc.indexes.create(
        name=INDEX_NAME,
        schema={"fields": {
            "embedding": {"type": "dense_vector", "dimension": int(vectors.shape[1]), "metric": "cosine"},
            "text": {"type": "string", "full_text_search": {}},
        }},
        deployment={"deployment_type": "managed", "cloud": "aws", "region": "us-east-1"},
    )
index = pc.index(INDEX_NAME)

records = []
for p in embedding_items:
    records.append({
        "_id": p["passage_id"], "embedding": passage_vectors[p["passage_id"]].tolist(),
        "text": p["text"], "source_id": p["source_id"], "version": p["version"],
        "status": p["status"], "access": p["access"], "model": p["model"] or "umum",
        "variant": "bad" if p in bad_r17 else "section", "run_id": RUN_ID,
    })
write_result = index.documents.upsert(namespace=NAMESPACE, documents=records)
assert write_result.upserted_count == len(records)

# Upsert Pinecone bersifat asynchronous. Tunggu sampai semua ID terlihat sebelum mencari.
record_ids = {p["passage_id"] for p in embedding_items}
for _ in range(15):
    fetched = index.documents.fetch(namespace=NAMESPACE, ids=sorted(record_ids), include_fields=["run_id"])
    if set(fetched.documents) == record_ids and all(doc.run_id == RUN_ID for doc in fetched.documents.values()):
        break
    time.sleep(2)
else:
    raise RuntimeError("Passage belum semuanya terlihat di Pinecone. Tunggu sebentar lalu jalankan ulang sel ini.")

def pinecone_filter(*, variant, active, model, source_id):
    '''Buat filter metadata sebelum Pinecone menghitung ranking.

    Jika active=False, arsip boleh ikut agar kesalahan pada Kasus 1 terlihat.
    '''
    terms = [
        {"run_id": {"$eq": RUN_ID}},
        {"variant": {"$eq": variant}},
        {"access": {"$in": sorted(ALLOWED_ACCESS)}},
    ]
    if active:
        terms.append({"status": {"$eq": "berlaku"}})
    if model:
        terms.append({"model": {"$eq": model}})
    if source_id:
        terms.append({"source_id": {"$eq": source_id}})
    return {"$and": terms}

def pinecone_search(*, case_id, method, limit, variant, active, model, source_id):
    '''Cari passage dengan dense search atau BM25 di Pinecone.

    Kembalikan passage, skor, dan urutan hasil; query selalu memakai filter metadata.
    '''
    if method == "dense":
        scoring = [DenseVectorQuery(field="embedding", values=query_vectors[case_id].tolist())]
    elif method == "bm25":
        scoring = [TextQuery(query=CASES[case_id]["question"], fields=["text"])]
    else:
        raise ValueError("method harus 'dense' atau 'bm25'")
    started = time.perf_counter()
    metadata_filter = pinecone_filter(
        variant=variant, active=active, model=model, source_id=source_id,
    )
    print("Filter Pinecone:", metadata_filter)
    response = index.documents.search(
        namespace=NAMESPACE, score_by=scoring, top_k=limit,
        filter=metadata_filter,
        include_fields=["source_id", "status", "access", "variant", "run_id"],
    )
    elapsed = time.perf_counter() - started
    results = []
    for match in response.matches:
        if match.id not in passage_by_id or match.run_id != RUN_ID:
            raise RuntimeError("Hasil Pinecone tidak cocok dengan korpus run ini. Periksa namespace dan index.")
        results.append({"passage": passage_by_id[match.id], "score": float(match.score)})
    print(f"Pinecone {method}: {elapsed:.3f}s | hasil: {len(results)}")
    return results

def source_rank(results, source_id):
    '''Cari peringkat pertama sebuah dokumen; hasil None berarti tidak ditemukan.'''
    return next((i for i, item in enumerate(results, 1) if item["passage"]["source_id"] == source_id), None)

def report(case_id, results):
    '''Tampilkan peringkat dokumen dan passage bukti untuk satu kasus.

    Hit@k hanya dihitung untuk kasus yang punya jawaban acuan di korpus.
    '''
    case = CASES[case_id]
    if case["source"] is None:
        print(case_id, "tidak punya jawaban biaya dalam korpus; Hit@k tidak dihitung")
        return
    doc_rank = source_rank(results, case["source"])
    passage_rank = evidence_rank(results, case["source"], case["evidence"])
    print(f"{case_id}: peringkat dokumen={doc_rank}, peringkat passage jawaban={passage_rank}, "
          f"Hit dokumen@1={doc_rank == 1}, Hit dokumen@3={doc_rank is not None and doc_rank <= 3}, "
          f"Hit passage@1={passage_rank == 1}, Hit passage@3={passage_rank is not None and passage_rank <= 3}")

print("Model:", EMBED_MODEL, "| input:", embedding_info["inputs"], "| dimensi:", vectors.shape[1])
print("API dipanggil pada run ini:", not embedding_info["cache_hit"],
      "| detik API:", round(embedding_info["api_seconds"], 2),
      "| token API:", embedding_info["api_tokens"])
print("Pinecone index:", INDEX_NAME, "| namespace:", NAMESPACE, "| passage:", len(records))

### Baca parameter pencariannya

Setiap contoh di bawah menulis semua parameter `pinecone_search(...)` supaya kamu bisa melihat persis apa yang berubah:

| Parameter | Artinya |
| --- | --- |
| `case_id` | Pertanyaan kasus yang dicari. |
| `method` | `dense` untuk embedding atau `bm25` untuk kecocokan teks. |
| `limit` | Jumlah maksimum passage dari Pinecone. |
| `variant` | `bad` untuk potongan tabel yang sengaja rusak; `section` untuk passage yang utuh. |
| `active` | `True` menambahkan filter `status=berlaku`; `False` membolehkan arsip ikut dicari. |
| `model` | ID model perangkat yang wajib cocok; `None` berarti tidak membatasi model. |
| `source_id` | ID dokumen yang wajib cocok; `None` berarti mencari di semua dokumen yang lolos filter lain. |

Fungsi pencarian juga mencetak filter yang benar-benar dikirim ke Pinecone. Pada perbandingan sebelum/sesudah, perhatikan parameter mana yang berubah dan mana yang tetap.

## Kasus 1 — Aturan lama ikut muncul

**Masalah:** Aturan pembatalan lama menyebut **48 jam**. Aturan yang sekarang berlaku menyebut **24 jam**. Jika kita mencari di semua dokumen tanpa memeriksa status, jawaban bisa mengutip aturan lama.

**Tebak dulu:** apa yang bisa terjadi jika dokumen arsip mendapat skor tertinggi? Tulis alasanmu, lalu jalankan sel berikut dan lihat status, versi, isi, serta posisi tiap hasil.

In [ ]:
case = CASES["C1"]
old = next(p for p in all_passages if p["source_id"] == "PEMBATALAN-ARSIP")
current = next(p for p in active_passages if p["source_id"] == "PEMBATALAN" and case["evidence"].casefold() in p["text"].casefold())
print("SIMULASI TERKENDALI — contoh jika arsip muncul di atas aturan yang berlaku:")
show_results([{"passage": old, "score": 2.0}, {"passage": current, "score": 1.0}])
print("\nHASIL RUN — Pinecone dense search tanpa filter status:")
unfiltered_c1 = pinecone_search(
    case_id="C1", method="dense", limit=5,
    variant="section", active=False, model=None, source_id=None,
)
show_results(unfiltered_c1)
report("C1", unfiltered_c1)

**Apa yang salah?** Dokumen arsip seharusnya tidak boleh menjadi sumber jawaban, seberapa pun tinggi skor kemiripannya. Saring `Status: berlaku` **sebelum** retrieval. Setelah disaring, cek bahwa arsip benar-benar hilang dari daftar kandidat. Peringkat dokumen aktif masih bisa berubah karena model embedding.

In [ ]:
eligible_c1 = [p for p in all_passages if p["status"] == "berlaku" and p["access"] in ALLOWED_ACCESS]
assert all(p["source_id"] != "PEMBATALAN-ARSIP" for p in eligible_c1)
print("HASIL RUN — Pinecone menerima filter status sebelum retrieval:")
filtered_c1 = pinecone_search(
    case_id="C1", method="dense", limit=5,
    variant="section", active=True, model=None, source_id=None,
)
assert all(item["passage"]["status"] == "berlaku" for item in filtered_c1)
show_results(filtered_c1)
report("C1", filtered_c1)
print("Cek: dokumen arsip sudah tidak bisa masuk hasil.")
print("Pelajaran C1: saring status dan akses sebelum retrieval. Skor hanya mengurutkan dokumen yang memang boleh dipakai.")

# Catatanmu: masalah awal = ... ; perbaikan = ... ; hasil run = ...

## Kasus 2 — Baris tabel R17 terpotong

**Masalah:** Pada manual KD-410, `R17` berarti “pembaca kartu akses tidak tersambung”. Cara memotong teks yang asal bisa memisahkan kode dan artinya. Hasil pencarian mungkin menunjuk dokumen yang benar, tetapi **tidak ada satu passage** yang berisi jawaban lengkap.

**Tebak dulu:** kalau sistem menemukan dokumen KD410-ERROR, apakah itu berarti jawaban sudah tersedia di satu passage? Lihat dua potongan yang sengaja dibuat buruk di bawah.

In [ ]:
print("SIMULASI TERKENDALI — satu baris tabel sengaja dipotong:")
for p in bad_r17:
    print(p["passage_id"], "→", p["text"])
    assert not ("R17" in p["text"] and CASES["C2"]["evidence"].casefold() in p["text"].casefold())
bad_results = pinecone_search(
    case_id="C2", method="dense", limit=2,
    variant="bad", active=True, model=None, source_id="KD410-ERROR",
)
print("HASIL RUN — ranking dua potongan buruk dari Pinecone:")
show_results(bad_results)
report("C2", bad_results)

**Perbaikan:** simpan satu baris tabel dalam satu passage. Sertakan juga judul dokumen dan judul bagiannya agar passage tetap punya konteks. Fungsi di `rag_helpers.py` melakukan ini untuk tabel dalam korpus. Pada perbandingan pertama, semua parameter pencarian tetap sama; **hanya `variant` berubah** dari `bad` ke `section`. Setelah itu, ulangi pencarian di seluruh korpus aktif sebagai uji yang lebih realistis.

In [ ]:
assert "R17" in good_r17["text"] and CASES["C2"]["evidence"].casefold() in good_r17["text"].casefold()
print("Passage utuh:", good_r17["passage_id"], "→", good_r17["text"])
print("HASIL RUN — cakupan sama, hanya variant berubah:")
fixed_c2 = pinecone_search(
    case_id="C2", method="dense", limit=2,
    variant="section", active=True, model=None, source_id="KD410-ERROR",
)
show_results(fixed_c2)
report("C2", fixed_c2)
print("\nHASIL RUN — baris utuh di seluruh korpus aktif:")
full_c2 = pinecone_search(
    case_id="C2", method="dense", limit=5,
    variant="section", active=True, model=None, source_id=None,
)
show_results(full_c2)
report("C2", full_c2)
print("Cek: kode dan artinya sekarang ada dalam satu passage.")
print("Pelajaran C2: ID dokumen yang benar belum cukup. Periksa apakah passage benar-benar memuat jawaban.")

# Catatanmu: apakah jawaban lengkap sebelum/sesudah? Apakah ranking juga berubah? ...

## Kasus 3 — R17, tetapi terminalnya berbeda

**Masalah:** KD-410 dan KD-420 sama-sama memiliki kode `R17`, tetapi artinya **berbeda**. Jika pertanyaannya menyebut KD-420, hasil dari KD-410 tidak boleh dipakai hanya karena kata-katanya mirip.

**Tebak dulu:** informasi apa yang harus cocok persis? Contoh simulasi di bawah sengaja menaruh KD-410 di atas KD-420. Lalu lihat hasil run dari dense search dan BM25; hasilmu bisa berbeda dari contoh.

In [ ]:
case = CASES["C3"]
wrong_model = good_r17
correct_model = next(p for p in active_passages if p["source_id"] == "KD420-ERROR" and case["evidence"].casefold() in p["text"].casefold())
print("SIMULASI TERKENDALI — hasil dari model terminal yang salah berada di atas:")
show_results([{"passage": wrong_model, "score": 2.0}, {"passage": correct_model, "score": 1.0}])

start = time.perf_counter()
dense_c3 = pinecone_search(
    case_id="C3", method="dense", limit=5,
    variant="section", active=True, model=None, source_id=None,
)
dense_seconds = time.perf_counter() - start
start = time.perf_counter()
bm25_c3 = pinecone_search(
    case_id="C3", method="bm25", limit=5,
    variant="section", active=True, model=None, source_id=None,
)
bm25_seconds = time.perf_counter() - start
print("\nHASIL RUN — Pinecone dense:")
show_results(dense_c3); report("C3", dense_c3)
print("\nHASIL RUN — Pinecone BM25:")
show_results(bm25_c3); report("C3", bm25_c3)

**Perbaikan:** Pinecone BM25 mencocokkan kata/kode seperti `KD-420`. RRF menggabungkan **urutan hasil** dari Pinecone dense search dan BM25, bukan menjumlahkan skor keduanya. Coba bandingkan: apakah hybrid benar-benar lebih baik untuk pertanyaan ini? Karena pertanyaan menyebut model yang spesifik, kamu juga bisa menyaring metadata `Model perangkat` langsung di query Pinecone.

Bandingkan posisi passage yang berisi jawaban pada tiga metode. Setelah menyaring model, pastikan KD-410 tidak bisa muncul. Pilih metode berdasarkan hasil yang kamu lihat, bukan berdasarkan asumsi bahwa hybrid pasti menang.

In [ ]:
def rrf(dense, lexical, constant=60):
    '''Gabungkan dua daftar hasil berdasarkan peringkat, bukan skor mentah.'''
    scores, by_id = {}, {}
    for results in (dense, lexical):
        for rank, item in enumerate(results, 1):
            pid = item["passage"]["passage_id"]
            scores[pid] = scores.get(pid, 0.0) + 1 / (constant + rank)
            by_id[pid] = item["passage"]
    ids = sorted(scores, key=lambda pid: (-scores[pid], pid))
    return [{"passage": by_id[pid], "score": scores[pid]} for pid in ids]

def hybrid_search(*, case_id, limit, variant, active, model, source_id):
    '''Gabungkan hasil dense dan BM25 dari Pinecone dengan RRF.

    Jika model diisi, kedua query hanya mencari manual perangkat itu.
    '''
    dense = pinecone_search(
        case_id=case_id, method="dense", limit=limit,
        variant=variant, active=active, model=model, source_id=source_id,
    )
    lexical = pinecone_search(
        case_id=case_id, method="bm25", limit=limit,
        variant=variant, active=active, model=model, source_id=source_id,
    )
    return rrf(dense, lexical)[:limit]

start = time.perf_counter()
hybrid_c3 = hybrid_search(
    case_id="C3", limit=5, variant="section",
    active=True, model=None, source_id=None,
)
hybrid_seconds = time.perf_counter() - start
print("HASIL RUN — RRF hybrid:")
show_results(hybrid_c3); report("C3", hybrid_c3)

model_c3 = [p for p in active_passages if p["model"] == "KD-420"]
assert model_c3 and all(p["model"] == "KD-420" for p in model_c3)
print("\nHASIL RUN — model perangkat difilter sebelum retrieval:")
filtered_c3 = hybrid_search(
    case_id="C3", limit=5, variant="section",
    active=True, model="KD-420", source_id=None,
)
assert all(item["passage"]["model"] == "KD-420" for item in filtered_c3)
show_results(filtered_c3); report("C3", filtered_c3)
print(f"Waktu pencarian Pinecone: dense {dense_seconds:.3f}s | BM25 {bm25_seconds:.3f}s | dua query hybrid {hybrid_seconds:.3f}s")
print("Pelajaran C3: bandingkan ranking hasil run. Filter model memastikan manual model lain tidak ikut dicari.")

# Catatanmu: metode mana yang membantu? Apa efek filter model? ...

## Kasus 4 — Dokumennya relevan, angka biayanya tidak ada

**Masalah:** Dokumen kartu hilang menjelaskan cara melapor, menonaktifkan kartu, dan meminta penggantian. Dokumen itu **tidak menyebut biaya penggantian**. Kalimat “biayanya Rp50.000” di bawah adalah **contoh jawaban salah**, bukan informasi RuangKarya.

**Tebak dulu:** jika `KARTU-HILANG` ada di peringkat pertama, apakah kita sudah boleh menjawab dengan angka biaya? Baca passage yang ditemukan dan cari angka tersebut.

In [ ]:
print("SIMULASI TERKENDALI — jawaban tanpa bukti: 'Biayanya Rp50.000.' (SALAH)")
print("HASIL RUN — passage terdekat untuk pertanyaan biaya:")
live_c4 = pinecone_search(
    case_id="C4", method="dense", limit=5,
    variant="section", active=True, model=None, source_id=None,
)
show_results(live_c4)
report("C4", live_c4)
lost_card_passages = [p for p in active_passages if p["source_id"] == "KARTU-HILANG"]
print("\nSemua passage KARTU-HILANG:")
for p in lost_card_passages:
    print(p["passage_id"], "→", p["text"])

**Perbaikan:** pertanyaan ini meminta **angka biaya**, jadi cari angka itu pada sumber yang tepat. Jika tidak ada, katakan bahwa informasi biaya belum tersedia di dokumen dan arahkan ke sumber tarif resmi. Dalam lab ini kita memakai pencarian pola `Rp` yang sederhana. Di aplikasi sungguhan, pemeriksaan dukungan bukti perlu lebih kuat daripada satu regex.

In [ ]:
fee_evidence = [
    p for p in lost_card_passages
    if re.search(r"Rp\s*[\d.]+", p["text"], flags=re.IGNORECASE)
]
assert not fee_evidence
answer_c4 = (
    "Informasi biaya penggantian kartu akses tidak tersedia dalam dokumen ini. "
    "Silakan periksa sumber tarif resmi. [KARTU-HILANG, versi 2026-09]"
)
print("Jawaban yang didukung bukti:", answer_c4)
print("Cek: dokumen KARTU-HILANG tidak menyebut angka biaya. C4 tidak dihitung dalam Hit@k.")
print("Pelajaran C4: dokumen yang relevan belum tentu memuat fakta yang diminta. Jangan mengarang angka.")

# Catatanmu: dokumen yang relevan = ... ; fakta yang belum ada = ... ; jawaban = ...

## Coba sendiri — 15 menit

Isi tabel dari **hasil run-mu**, bukan dari contoh simulasi. Untuk C1–C3, catat posisi passage yang berisi jawaban serta `Hit@1`/`Hit@3` sebelum dan sesudah perubahan. Untuk C4, jelaskan mengapa kamu menahan jawaban. Waktu OpenAI embedding dan query Pinecone dicatat terpisah.

| Kasus | Di mana masalahnya? | Posisi bukti sebelum → sesudah | Apa kesimpulanmu? |
| --- | --- | --- | --- |
| C1 — arsip | … | … | … |
| C2 — tabel | … | … | … |
| C3 — model perangkat | … | … | … |
| C4 — tanpa biaya | … | Tidak ada angka biaya | … |

**Tugas mandiri:** jawab pertanyaan baru: “Berapa lama tombol reset PD-210 harus ditahan?” Gunakan pencarian dengan parameter lengkap, tunjukkan passage yang memuat jawabannya, dan sertakan ID sumber. Jelaskan apakah kamu perlu menyaring `model` atau `source_id` agar jawaban tidak tertukar dengan PD-220.

**Kumpulkan:** notebook yang sudah kamu jalankan, tabel di atas, satu contoh teknik yang **tidak** memperbaiki ranking (jika ada), dan jawaban C4 tanpa angka karangan. Nilai acuan di `CASES` hanya untuk mengecek hasil lab; jangan pakai nilai itu untuk memaksa jawaban aplikasi.

**Opsional:** gunakan model jawaban yang tersedia di akun OpenAI-mu untuk menulis jawaban dari tiga passage teratas. Kirim hanya passage yang lolos filter status dan akses, minta sitasi, lalu periksa setiap klaimnya. Pada C4, model harus mengatakan bahwa angka biaya tidak tersedia. Embedding API sendiri hanya membuat vektor; API itu tidak menulis jawaban.